# Download and create data

/data folder contain a already merged and preprocessed data.
But if you want to go to the original sources and how is it filtered out or merged, you can rely on it or use this notebook to understand how and why.
This is a needed step if you want to run this framework with other dataset.

- mergedcounts_generation.py
- metadata_generation.py



## Running against your own sandbox copy

The cells below never touch `data/` directly. They:

1. Copy just the inputs these two scripts read from `data/` (`metadata.csv`, `mergedcounts.csv`, and the `ucam_sanyal/counts_matrix.csv` duplicate used for cross-checking) into a **fresh** `sandbox/data_dev/` folder -- wiping and recreating it every time the first cell runs, so you always start from a clean copy.
2. Point `mergedcounts_generation.py` / `metadata_generation.py` at that copy via the `MASLD_DATA_DIR` environment variable (both scripts fall back to the real `data/` when it's unset) and import them as modules.
3. Call their `main()` functions against `sandbox/data_dev/` -- any `--write` output lands there, never in `data/`.

`sandbox/data_dev/` is already gitignored, so nothing produced here is ever committed. Re-run the first cell any time you want to start over.


In [ ]:
import importlib
import os
import shutil
import sys
from pathlib import Path

# Locate the repo root regardless of where Jupyter was launched from
# (e.g. `pixi run jupyter lab notebooks/` from sandbox/, per sandbox/CLAUDE_NOTES.md).
REPO_ROOT = Path.cwd().resolve()
for _ in range(6):
    if (REPO_ROOT / "src").is_dir() and (REPO_ROOT / "data").is_dir():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise RuntimeError(
        "Could not locate the repo root (a directory containing both src/ and "
        "data/) above the current working directory."
    )

DATA_DIR = REPO_ROOT / "data"
DATA_DEV_DIR = REPO_ROOT / "sandbox" / "data_dev"
SCRIPTS_DIR = REPO_ROOT / "src" / "00.download_and_merge_data"

# Files mergedcounts_generation.py / metadata_generation.py read from DATA_DIR.
# Add to this list as later sandbox steps need more inputs copied from data/.
SEED_FILES = [
    "metadata.csv",
    "mergedcounts.csv",
    "ucam_sanyal/counts_matrix.csv",  # known-duplicate cross-check
]


def fresh_data_dev() -> None:
    """Wipe and recreate sandbox/data_dev/, seeded with copies of the source
    files these two scripts need from data/. Safe to re-run any time you
    want a clean copy -- data/ itself is only ever read, never modified."""
    if DATA_DEV_DIR.exists():
        shutil.rmtree(DATA_DEV_DIR)
    DATA_DEV_DIR.mkdir(parents=True)
    for rel_path in SEED_FILES:
        src = DATA_DIR / rel_path
        dest = DATA_DEV_DIR / rel_path
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)
    print(f"[OK] Seeded {DATA_DEV_DIR} with {len(SEED_FILES)} file(s) copied from {DATA_DIR}.")


fresh_data_dev()

# Point the scripts at sandbox/data_dev/ instead of data/, then (re)import them
# so their module-level path constants pick up the new MASLD_DATA_DIR.
os.environ["MASLD_DATA_DIR"] = str(DATA_DEV_DIR)
sys.path.insert(0, str(SCRIPTS_DIR))

import mergedcounts_generation
import metadata_generation

importlib.reload(mergedcounts_generation)
importlib.reload(metadata_generation)

print(f"[OK] mergedcounts_generation.DATA_DIR = {mergedcounts_generation.DATA_DIR}")
print(f"[OK] metadata_generation.DATA_DIR     = {metadata_generation.DATA_DIR}")


In [ ]:
# Validates the seeded sandbox/data_dev/mergedcounts.csv against
# sandbox/data_dev/metadata.csv, then rewrites it with sample-name columns.
# data/mergedcounts.csv itself is never touched (MASLD_DATA_DIR redirected
# DATA_DIR to sandbox/data_dev/ above).
mergedcounts_generation.main(write=True)


In [ ]:
# Cross-checks sandbox/data_dev/metadata.csv against the public GEO/ArrayExpress
# records (network access required). Never writes -- see the script's
# docstring for why metadata.csv can't be fully regenerated from scratch.
metadata_generation.main()


`sandbox/data_dev/` now holds an independently validated/regenerated copy of `mergedcounts.csv` alongside the seeded `metadata.csv` -- inspect it, diff it against `data/`, or just re-run `fresh_data_dev()` above to blow it away and start over.


## Optional: true from-scratch regeneration from raw FASTQ (`regenerate_from_raw_fastq.py`)

**This is a separate, independent script -- not a later stage of the two above.** It produces the *complete* final matrix by itself; you do not run `mergedcounts_generation.py` afterward to "finish" it.

Why a third script, when `mergedcounts_generation.py` already claims to "merge from raw counts"? Because that script's real merge path only fires if you already have the two cohorts' separate per-sample HTSeq count matrices sitting locally -- nobody does, in this repo, so in practice it always falls back to validating/relabelling the already-committed `data/mergedcounts.csv`. It never touches actual sequencing reads.

`regenerate_from_raw_fastq.py` is the one that does: it downloads the real raw paired-end FASTQ for both cohorts (UCAM via ArrayExpress E-MTAB-9815/ENA, VCU/Sanyal via GEO GSE130970 -> SRA SRP197353), runs each sample through FastQC -> HISAT2 (GRCh38) -> HTSeq itself, then assembles, inner-joins, and filters the two cohorts on its own -- the same steps `mergedcounts_generation.py` would run *if* it had the raw material. It writes to a **different file**, `mergedcounts_from_raw_fastq.csv` (never `mergedcounts.csv`), specifically so it can't silently overwrite or be confused with the committed one -- treat it as an independent reproduction to diff against, not a drop-in replacement.

**When to actually reach for it:** essentially never for routine work. Use `mergedcounts_generation.py` above for everyday reproducibility -- it reproduces the committed table exactly, in seconds, no downloads. Use `regenerate_from_raw_fastq.py` only if you specifically want to verify the paper's pipeline end-to-end starting from the public raw reads, or need to run a *different* raw-FASTQ cohort through the same steps. Real cost: ~338 GB of downloads and, per an informal (unbenchmarked-on-any-specific-machine) estimate, **4-12+ days** of alignment compute on a machine with ~8 GB RAM -- see the script's own module docstring for the full breakdown, including two undocumented-in-the-paper assumptions it has to make (Ensembl annotation release, library strandedness) that mean it is not expected to be byte-identical to `data/mergedcounts.csv`.

The cell below only runs `--dry-run` -- it fetches the real sample manifest (network reads only) and prints sizes/read counts, no downloads, no bioinformatics tools required. `--benchmark` (times 1 sample, extrapolates a real estimate for your machine) and `--full` (the actual multi-day run) are deliberately not run from this notebook -- see the commands underneath.


In [ ]:
import subprocess

# Cheap sanity check only: fetches the real sample manifest from ArrayExpress/
# ENA + GEO/SRA and prints sizes/read counts. No downloads, no tools required.
subprocess.run(
    [sys.executable, str(SCRIPTS_DIR / "regenerate_from_raw_fastq.py"), "--dry-run"],
    env={**os.environ, "MASLD_DATA_DIR": str(DATA_DEV_DIR)},
    check=True,
)


To actually benchmark or run the real regeneration (outside this notebook, from `src/00.download_and_merge_data/`, ideally on a machine with well over 8 GB RAM):

```bash
# Times 1 sample end-to-end, extrapolates a real estimate for this machine:
MASLD_DATA_DIR=../../sandbox/data_dev python regenerate_from_raw_fastq.py --benchmark

# The real, multi-day, ~338 GB run:
MASLD_DATA_DIR=../../sandbox/data_dev python regenerate_from_raw_fastq.py --full --parallel 4
```

Both require `fastqc`, `hisat2`, `samtools`, and `htseq-count` on `PATH` (e.g. `pixi add -c bioconda -c conda-forge fastqc hisat2 samtools htseq`) -- neither is installed in this sandbox env yet.
